In [1]:
import numpy as np
import pandas as pd
from nlp.utils import load_embedder, cosine_similarity

# Message Intent Classification

In [2]:
from nlp.pipelines import MessageIntentPipeline
from nlp.rubric import MessageIntentRubric
golden_set_intent_df = pd.read_csv('../data/processed/golden_set_intents.csv').set_index('message_id')

In [3]:
message_intent_rubric = MessageIntentRubric(
    hypothesis_template="The user is {}.",
    intent_labels={
        "asking to create or make a new video, or providing a script, brief, or topic for one": "Create",
        "asking to change, revise, or adjust an existing video, including specifying what its script should say or how it should look": "Edit",
        "confirming the video looks good, appreciating the video, or approving it as final": "Approval",
        "pointing out that the assistant did something wrong or made an unwanted change": "Dissatisfaction",
    },
    edit_intent_labels={
        "editing the spoken script, narration, or wording": "Edit content",
        "editing the visuals: avatar, background, colour, font, layout, template, "
        "captions, overlay, slides, transition, or animation": "Edit visual",
    },
)
message_intent_clf = MessageIntentPipeline(model_name="facebook/bart-large-mnli", rubric=message_intent_rubric)
predicted_intent_df = message_intent_clf.predict(golden_set_intent_df['content'])

/Users/ridho.a/personal-projects/projects/conversation-data-nlp-video-generation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 515/515 [00:00<00:00, 9132.51it/s]


In [4]:
from sklearn.metrics import classification_report
labels = ["Create", "Edit content", "Edit visual", "Approval", "Dissatisfaction"]
y_true = golden_set_intent_df["intent_human"]
y_pred = predicted_intent_df["intent"]
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

                 precision    recall  f1-score   support

         Create       1.00      1.00      1.00        10
   Edit content       0.76      0.76      0.76        17
    Edit visual       0.92      0.79      0.85        14
       Approval       0.83      1.00      0.91        10
Dissatisfaction       0.83      0.83      0.83        12

       accuracy                           0.86        63
      macro avg       0.87      0.88      0.87        63
   weighted avg       0.86      0.86      0.86        63



In [5]:
# Apply to all messages
messages_df = pd.read_csv('../data/processed/messages.csv')
user_messages_df = (
    messages_df[messages_df['role']=='user'].copy()
    .set_index('message_id')
)
user_messages_intent_df = (
    message_intent_clf.predict(user_messages_df['content'], progress=True)
    .reset_index()
)

intent · stage 2 (edits): 100%|███████████████████████████████████████████████████| 110/110 [01:19<00:00,  1.38batch/s]


In [6]:
user_messages_intent_df.to_csv('../data/processed/user_messages_intent.csv', index=False)
user_messages_intent_df.head()

,message_id,intent,intent_confidence,edit_intent_split_confidence
0,msg_001117,Create,0.618346,NaN
1,msg_001119,Edit content,0.734213,0.876979
2,msg_012149,Create,0.767021,NaN
3,msg_012151,Edit content,0.707586,0.801053
4,msg_012153,Edit content,0.621103,0.972719


In [7]:
user_messages_intent_df.groupby(['intent']).agg(
    num_messages=('intent', 'count'),
    avg_confidence=('intent_confidence', 'mean'),
    avg_edit_intent_split_confidence=('edit_intent_split_confidence', 'mean'),
)

,num_messages,avg_confidence,avg_edit_intent_split_confidence
intent,,,
Approval,604,0.598501,NaN
Create,1153,0.608538,NaN
Dissatisfaction,977,0.688965,NaN
Edit content,2296,0.707685,0.854194
Edit visual,1205,0.605475,0.837089
Other,136,1.000000,NaN


# Brief Anatomy Completeness Check

In [8]:
from nlp.pipelines import FullBriefCheckerPipeline
from nlp.rubric import FullBriefCheckerKeywords 
golden_set_brief_checklist_df = pd.read_csv('../data/processed/golden_set_brief_checklist.csv').set_index('message_id')

In [9]:
full_brief_checker_keywords = FullBriefCheckerKeywords(
    avatar_names=("Sam", "Casey", "Riley", "Drew", "Alex", "Jordan", "Morgan", "Taylor"),
    avatar_label_cues=("avatar", "presenter", "speaker"),
    visual_terms=(
        "scene", "background", "transition", "font", "layout", "colour", "color",
        "logo", "overlay", "shot", "frame", "montage", "split screen", "split-screen",
        "screen", "animation", "graphic", "infographic", "visual",
    ),
    tone_terms=("professional",  "formal", "corporate", "serious", "fun", "casual", "friendly"),
    script_cues=("script:",),
    min_labels_for_structured=2,
)
full_brief_checker_clf = FullBriefCheckerPipeline(keywords=full_brief_checker_keywords)
predicted_brief_checklist_df = full_brief_checker_clf.extract(golden_set_brief_checklist_df['content'])

In [10]:
from sklearn.metrics import classification_report, accuracy_score

elements = ["has_script", "has_avatar", "has_visual", "has_tone", "structured"]
Y_true = golden_set_brief_checklist_df[elements].astype(int).values      
Y_pred = predicted_brief_checklist_df[elements].astype(int).values

print(classification_report(Y_true, Y_pred, target_names=elements, zero_division=0))
print("subset (exact-match) accuracy:", accuracy_score(Y_true, Y_pred))  # all 5 correct

              precision    recall  f1-score   support

  has_script       1.00      1.00      1.00         6
  has_avatar       1.00      1.00      1.00         8
  has_visual       1.00      1.00      1.00        10
    has_tone       1.00      1.00      1.00         2
  structured       1.00      1.00      1.00        11

   micro avg       1.00      1.00      1.00        37
   macro avg       1.00      1.00      1.00        37
weighted avg       1.00      1.00      1.00        37
 samples avg       0.86      0.86      0.86        37

subset (exact-match) accuracy: 1.0


In [11]:
# Apply to all messages where intent=Create (from message-intent clf in previous step)
user_messages_intent_df = pd.read_csv('../data/processed/user_messages_intent.csv')
create_intent_messages_df = (
    user_messages_intent_df[user_messages_intent_df['intent']=='Create'][['message_id','intent']]
    .merge(
        right=messages_df[['message_id', 'content']]
        , on=['message_id']
    )
    .set_index('message_id')
)
create_intent_messages_anatomy_df = (
    full_brief_checker_clf.extract(create_intent_messages_df['content'])
    .reset_index()
)

create_intent_messages_anatomy_df.to_csv('../data/processed/create_intent_messages_anatomy.csv', index=False)
create_intent_messages_anatomy_df.head()

,message_id,has_script,has_avatar,has_visual,has_tone,structured,completeness_score
0,msg_001117,False,False,False,False,False,0
1,msg_012149,False,False,False,False,False,0
2,msg_007961,True,False,False,False,False,1
3,msg_006437,False,False,True,True,True,3
4,msg_009941,True,False,False,False,False,1


# End of notebook –––––––––––––––––––––––––––––––––––––––––––––––––––––––––––